# 🏆 Notebook 10 — Capstone: Weather Data Analysis

> **Module:** Capstone Project · **Estimated time:** 60–90 min · **Difficulty:** Intermediate · **Prerequisites:** Notebooks 1–9

This is your **integration project**. You will use *everything* from the previous notebooks: variables, lists, dictionaries, functions, NumPy, pandas, matplotlib, and a sprinkling of scikit-learn. The deliverable is a small but complete analytical report on a year of weather data across five US cities.

## 🎯 What you will accomplish

1. **Generate** a realistic, reproducible weather dataset.
2. **Explore** it with descriptive statistics.
3. **Analyse** temperature and precipitation by city, month, and season.
4. **Visualise** the findings as a four-panel dashboard.
5. **Model** the relationship between temperature and precipitation with linear regression.
6. **Write** a short executive summary of what you learned.

By the end, you should be able to describe the dataset like a data scientist would in a Monday-morning briefing: *which cities are hottest, which are wettest, when, and how strong are the patterns?*

## 🧰 Tools you will use

- pandas DataFrames and `groupby`
- NumPy arrays
- matplotlib for the 2×2 dashboard
- scikit-learn for a quick regression
- f-strings everywhere

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
RNG = np.random.default_rng(seed=42)

plt.rcParams.update({
    "figure.figsize"   : (8, 5),
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
})

print(f"pandas {pd.__version__}, numpy {np.__version__}")

## 2. Build the dataset

In a real project you would load a CSV with `pd.read_csv("weather.csv")`. Here we **simulate** one so the notebook is fully self-contained and reproducible.

Each row is one (city, month) observation with:

| Column          | Meaning                                              |
|-----------------|------------------------------------------------------|
| `city`          | one of 5 US cities                                   |
| `month`, `month_num` | Jan, Feb, …, Dec  +  1–12                        |
| `temperature`   | mean daily °C for that month                          |
| `precipitation` | total mm of rain/snow for that month                  |
| `humidity`      | average % relative humidity                           |
| `wind`          | average km/h wind speed                               |

In [ ]:
cities = ["New York", "Los Angeles", "Chicago", "Miami", "Seattle"]
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Realistic monthly mean temperatures (°C) for each city
city_temps = {
    "New York":    [ 0,  2,  7, 13, 18, 23, 26, 25, 21, 15,  9,  3],
    "Los Angeles": [14, 14, 15, 17, 18, 20, 22, 23, 22, 19, 16, 14],
    "Chicago":     [-4, -2,  3, 10, 16, 22, 25, 24, 19, 12,  4, -2],
    "Miami":       [20, 21, 23, 25, 27, 28, 29, 29, 28, 26, 23, 21],
    "Seattle":     [ 5,  7,  9, 11, 14, 17, 20, 20, 17, 12,  8,  5],
}

# Average monthly precipitation (mm) per city
city_precip = {
    "New York":    [85, 75, 95, 100, 110, 95, 110, 100, 95, 85, 90, 90],
    "Los Angeles": [80, 90, 60, 25,   5,  2,   1,  2, 10, 20, 35, 60],
    "Chicago":     [50, 50, 70, 90,  100, 110, 95, 110, 85, 75, 65, 60],
    "Miami":       [50, 50, 55, 80,  130, 230, 165, 215, 230, 165, 80, 50],
    "Seattle":     [140,95,90, 75,   50,  40,  20, 25, 45, 90, 155, 145],
}

records = []
for city in cities:
    for month_idx, month in enumerate(months):
        # Add slight noise so values are not perfectly textbook
        t = city_temps[city][month_idx]   + RNG.normal(0, 1.0)
        p = city_precip[city][month_idx]  + RNG.normal(0, 5)
        h = 55 + RNG.normal(0, 8) + (5 if p > 100 else 0)
        w = 14 + RNG.normal(0, 3)
        records.append({
            "city":          city,
            "month":         month,
            "month_num":     month_idx + 1,
            "temperature":   round(float(t), 1),
            "precipitation": round(max(0.0, float(p)), 1),
            "humidity":      round(float(h), 1),
            "wind":          round(float(w), 1),
        })

df = pd.DataFrame(records)
print(f"Shape: {df.shape}")
df.head()

## 3. Quick exploration

In [ ]:
print("--- Schema and missing values ---")
df.info()
print()
print("--- Numeric summary ---")
df.describe().round(2)

In [ ]:
# Always check class / category balance
print(df["city"].value_counts())
print()
print(f"Cities  : {df['city'].nunique()}")
print(f"Months  : {df['month'].nunique()}")

All 5 cities × 12 months = 60 rows. Every column is fully populated. The numeric ranges look plausible — let\'s move on.

## 4. Temperature analysis

### 4.1 Average temperature by city

In [ ]:
by_city = df.groupby("city").agg(
    mean_temp=("temperature", "mean"),
    min_temp =("temperature", "min"),
    max_temp =("temperature", "max"),
    temp_range=("temperature", lambda s: s.max() - s.min()),
).round(2).sort_values("mean_temp", ascending=False)

by_city

> 🎯 **Observation.** Miami is hottest (warmest mean, warmest minimum). Chicago has the largest annual range — classic continental climate. Los Angeles is the *most stable* — small temperature range. That\'s the kind of insight a 5-second look at a table can already give us.

### 4.2 Seasonal grouping

Let\'s tag each row with a *season* using a helper function and `apply`.

In [ ]:
def season_of(month_num: int) -> str:
    """Map month number (1-12) to a meteorological season label."""
    if month_num in (12, 1, 2):  return "Winter"
    if month_num in (3, 4, 5):   return "Spring"
    if month_num in (6, 7, 8):   return "Summer"
    return "Autumn"

df["season"] = df["month_num"].apply(season_of)

# Mean temperature per city, per season — laid out as a tidy matrix
season_temp = df.pivot_table(
    index="city", columns="season", values="temperature", aggfunc="mean",
).round(1)

# Reorder seasons to the natural sequence
season_temp = season_temp[["Winter", "Spring", "Summer", "Autumn"]]
season_temp

## 5. Precipitation analysis

In [ ]:
precip_summary = df.groupby("city").agg(
    annual_precip=("precipitation", "sum"),
    mean_monthly =("precipitation", "mean"),
    wettest_month=("precipitation", "max"),
    driest_month =("precipitation", "min"),
).round(1).sort_values("annual_precip", ascending=False)

precip_summary

In [ ]:
# Which month is the wettest for each city?
wettest = (df.loc[df.groupby("city")["precipitation"].idxmax(),
                  ["city", "month", "precipitation"]]
             .rename(columns={"month": "wettest_month_name", "precipitation": "mm"}))
wettest.set_index("city")

### 5.1 A custom classifier for monthly rainfall

Wrap a tiny function to label each month\'s precipitation level. This is exactly the kind of *categorical feature engineering* you do all the time in real ML projects.

In [ ]:
def precip_category(mm: float) -> str:
    """Bucket monthly rainfall into qualitative categories."""
    if mm < 25:  return "Dry"
    if mm < 75:  return "Moderate"
    if mm < 150: return "Wet"
    return "Very wet"

df["precip_cat"] = df["precipitation"].apply(precip_category)

# How many months of each kind, per city?
df.pivot_table(index="city", columns="precip_cat",
               values="month", aggfunc="count", fill_value=0)

## 6. The dashboard

A 2×2 figure that conveys the four most important findings at a glance.

In [ ]:
palette = {"New York": "#4C72B0", "Los Angeles": "#DD8452",
           "Chicago": "#55A467", "Miami": "#C44E52", "Seattle": "#8172B2"}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Weather analysis dashboard — 5 US cities", fontsize=15, fontweight="bold")

# (0, 0) Temperature curves per city
for city in cities:
    sub = df[df["city"] == city].sort_values("month_num")
    axes[0, 0].plot(sub["month_num"], sub["temperature"],
                    marker="o", label=city, color=palette[city])
axes[0, 0].set_title("Monthly mean temperature")
axes[0, 0].set_xlabel("Month")
axes[0, 0].set_ylabel("°C")
axes[0, 0].set_xticks(range(1, 13))
axes[0, 0].set_xticklabels(months, rotation=30, fontsize=9)
axes[0, 0].legend(fontsize=9, loc="lower center", ncol=3)

# (0, 1) Total annual precipitation per city
totals = df.groupby("city")["precipitation"].sum().sort_values()
bar_colors = [palette[c] for c in totals.index]
axes[0, 1].barh(totals.index, totals.values, color=bar_colors, edgecolor="black")
for i, v in enumerate(totals.values):
    axes[0, 1].text(v + 10, i, f"{v:.0f} mm", va="center", fontsize=9)
axes[0, 1].set_title("Annual precipitation")
axes[0, 1].set_xlabel("Total mm")
axes[0, 1].set_xlim(0, totals.max() * 1.18)

# (1, 0) Temperature distribution per city — box plot
city_order = list(by_city.index)             # warm → cold
box_data = [df.loc[df["city"] == c, "temperature"].values for c in city_order]
bp = axes[1, 0].boxplot(box_data, tick_labels=city_order, patch_artist=True,
                         medianprops=dict(color="black", linewidth=2))
for patch, c in zip(bp["boxes"], city_order):
    patch.set_facecolor(palette[c]); patch.set_alpha(0.7)
axes[1, 0].set_title("Temperature distribution per city")
axes[1, 0].set_ylabel("°C")
axes[1, 0].tick_params(axis="x", rotation=20, labelsize=9)

# (1, 1) Temperature vs precipitation — coloured by city
for city in cities:
    sub = df[df["city"] == city]
    axes[1, 1].scatter(sub["temperature"], sub["precipitation"],
                       s=55, label=city, color=palette[city], edgecolor="black", alpha=0.8)
axes[1, 1].set_title("Temperature vs precipitation (monthly)")
axes[1, 1].set_xlabel("Temperature (°C)")
axes[1, 1].set_ylabel("Precipitation (mm)")
axes[1, 1].legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.show()

**Reading the dashboard.**

- **Top-left.** Miami is hottest year-round; Chicago and New York show the deepest seasonal cycles; Seattle is mild.
- **Top-right.** Miami also receives the most total rain, followed by New York and Seattle; Los Angeles is by far the driest.
- **Bottom-left.** Box widths show the **annual range**: LA and Miami are tight; Chicago is widest.
- **Bottom-right.** When you colour points by city, you see distinct *climate clusters*. Miami sits high (hot + wet); LA hugs the bottom (warm + dry).

## 7. A statistical question — is temperature predictive of rain?

Across *all* cities and months, is there a relationship between temperature and precipitation? Let\'s answer two ways: a simple correlation, and a one-feature linear regression.

In [ ]:
# Global correlation (Pearson)
overall_corr = df["temperature"].corr(df["precipitation"])
print(f"Pearson correlation (all data) : r = {overall_corr:+.3f}")

# By city — does each city have its own pattern?
print("\nPer-city correlation:")
for city, sub in df.groupby("city"):
    r = sub["temperature"].corr(sub["precipitation"])
    print(f"  {city:<12} r = {r:+.3f}")

Different cities show different patterns: in **Miami** temperature and rainfall rise together (summer thunderstorm season); in **LA** they go in opposite directions (a dry-summer Mediterranean climate). The global correlation is therefore misleading — a great example of *Simpson\'s paradox*. **Always group by category before drawing conclusions.**

In [ ]:
# Quick linear regression: predict precipitation from temperature, per city
from sklearn.linear_model import LinearRegression

fig, ax = plt.subplots(figsize=(9, 5))
for city in cities:
    sub = df[df["city"] == city]
    X = sub[["temperature"]].values
    y = sub["precipitation"].values
    model = LinearRegression().fit(X, y)
    xs = np.linspace(X.min(), X.max(), 30).reshape(-1, 1)
    ys = model.predict(xs)

    ax.scatter(X, y, color=palette[city], edgecolor="black", alpha=0.7, label=None)
    ax.plot(xs, ys, color=palette[city], linewidth=2,
            label=f"{city}: slope = {model.coef_[0]:+.2f} mm/°C")

ax.set_title("Per-city linear fit: precipitation vs temperature")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Precipitation (mm)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The slopes confirm the story.** Miami has a strongly positive slope (more heat → more rain), Los Angeles a strongly negative slope (hot summers are dry), and the others are somewhere in between.

## 8. Bonus — climate similarity

Which cities have the most similar *climate*? Build a small distance matrix using each city\'s monthly temperature profile.

In [ ]:
# Pivot to a city × month matrix
profiles = df.pivot_table(index="city", columns="month_num", values="temperature")
# Standardise each city so we compare *shape*, not absolute level (optional)
# profiles = (profiles.subtract(profiles.mean(axis=1), axis=0))

# Euclidean distance between every pair of cities
from itertools import combinations
distances = pd.DataFrame(index=cities, columns=cities, dtype=float)
for a, b in combinations(cities, 2):
    d = float(np.linalg.norm(profiles.loc[a].values - profiles.loc[b].values))
    distances.loc[a, b] = d
    distances.loc[b, a] = d
distances = distances.fillna(0).round(1)

print("Pairwise distance of monthly temperature profiles (lower = more similar):")
print(distances)

In [ ]:
# Heatmap of the similarity matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(distances.values, cmap="viridis")
ax.set_xticks(range(len(cities)), cities, rotation=20)
ax.set_yticks(range(len(cities)), cities)
for i in range(len(cities)):
    for j in range(len(cities)):
        ax.text(j, i, f"{distances.iloc[i, j]:.0f}", ha="center", va="center",
                color="white" if distances.iloc[i, j] > distances.values.max()/2 else "black")
ax.set_title("Distance between cities' temperature profiles")
fig.colorbar(im, ax=ax, label="Euclidean distance")
plt.tight_layout()
plt.show()

Chicago and New York are climate twins; LA is the odd one out. (You could plug the same matrix into a clustering algorithm — that\'s how grouping by climate type actually works in practice.)

## 9. Your turn — write the executive summary

Imagine the next thing in the document is a 5-bullet summary for an executive who has 30 seconds. Drafting this is half the data-science job.

> _One bullet per insight. Concrete numbers. No buzzwords._

Try writing yours in the cell below, then expand the solution to compare.

In [ ]:
# Your 5 bullets here, in plain text  👇
summary = """
1. ...
2. ...
3. ...
4. ...
5. ...
"""
print(summary)

<details>
<summary>💡 <b>One possible summary</b></summary>

```
1. Miami is the hottest city year-round (mean ≈ 25 °C) and also the wettest (annual rainfall ≈ 1 500 mm), driven by a strong summer rainy season.
2. Chicago shows the largest annual temperature swing (~29 °C between coldest and hottest month) — a textbook continental climate.
3. Los Angeles is the driest of the five (annual rainfall < 400 mm) and the most thermally stable.
4. Temperature and precipitation are correlated *per city* but in opposite directions: positive in Miami, negative in LA — so the global correlation is misleading.
5. Chicago and New York have the most similar monthly temperature profiles; LA is the outlier.
```
</details>

## 🧪 Bonus exercises

### Exercise A — Add a "comfort index"

Build a new column `comfort_index = temperature - 0.1 * precipitation - 0.05 * humidity + 0.05 * wind`. For each city, find the *most comfortable* month according to this index.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
df["comfort_index"] = (df["temperature"]
                       - 0.1 * df["precipitation"]
                       - 0.05 * df["humidity"]
                       + 0.05 * df["wind"])

best = (df.loc[df.groupby("city")["comfort_index"].idxmax(),
               ["city", "month", "comfort_index"]]
        .set_index("city"))
print(best.round(2))
```
</details>

### Exercise B — Save the report

Save the final cleaned DataFrame (with `season`, `precip_cat`, `comfort_index`) as a CSV file called `weather_report.csv` using `df.to_csv(...)`. Then read it back to verify.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
df.to_csv("weather_report.csv", index=False)

# Read it back
loaded = pd.read_csv("weather_report.csv")
print(f"Saved & reloaded shape: {loaded.shape}")
loaded.head()
```
</details>

### Exercise C — Try a different prediction problem

Train a `RandomForestRegressor` to predict **precipitation** from `temperature`, `humidity`, `wind` and one-hot encoded `city`. Use 80/20 split, report MAE and R².

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X = pd.get_dummies(df[["temperature", "humidity", "wind", "city"]],
                   columns=["city"], drop_first=False)
y = df["precipitation"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(Xtr, ytr)
yhat = rf.predict(Xte)

print(f"MAE : {mean_absolute_error(yte, yhat):.2f} mm")
print(f"R²  : {r2_score(yte, yhat):.3f}")
```
</details>

## 🧠 Project takeaways

1. **A good analysis follows a story.** Load → explore → analyse → visualise → model → summarise.
2. **`groupby` and `pivot_table` are 80 % of pandas work** — master them.
3. **A 2×2 dashboard is enough** for an executive-level overview of most projects.
4. **Look at sub-groups before drawing global conclusions** (Simpson\'s paradox is real).
5. Even a simple linear regression is a *very* powerful first model when paired with thoughtful features.
6. Save your cleaned data as CSV — your future self will thank you.

## 🚀 What's next?

Congratulations — you have completed a full data-science project! Suggested next steps:

- **Real datasets.** Try Kaggle\'s [Titanic](https://www.kaggle.com/c/titanic), [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques), or the [UCI ML Repository](https://archive.ics.uci.edu/ml/index.php).
- **Deeper ML.** Cross-validation, regularisation (Ridge / Lasso), gradient boosting (XGBoost, LightGBM).
- **Deep learning.** TensorFlow / PyTorch for images, text, and time series.
- **Deployment.** Wrap your model into an API with FastAPI or Streamlit.
- **Storytelling.** Practise turning analyses into clear written reports and slide decks — communication is half the value.

> *"The best way to learn data science is to do data science."*